# Chapter 13: Databricks Mosaic AI - Deep Dive

This notebook provides extended, runnable examples demonstrating the core capabilities of Databricks Mosaic AI. 

*Note: Executing these cells requires an active Databricks workspace with Unity Catalog, Model Serving endpoints enabled, and appropriate permissions.*

## 1. Setup and Authentication

To interact with Databricks AI APIs locally or in a notebook, ensure you have the `databricks-sdk` and `mlflow` libraries installed.

In [ ]:
!pip install databricks-sdk mlflow langchain langchain-community databricks-vectorsearch

## 2. Mosaic AI Model Serving (Foundation Models)

Databricks provides built-in Foundation Model APIs (e.g., `databricks-dbrx-instruct`, `databricks-llama-2-70b-chat`). We can query these directly using the `mlflow.deployments` client.

In [ ]:
from mlflow.deployments import get_deploy_client

# Initialize the client. In a Databricks notebook, it automatically picks up the workspace context.
# Locally, ensure DATABRICKS_HOST and DATABRICKS_TOKEN environment variables are set.
try:
    client = get_deploy_client("databricks")
    print("Deploy client initialized successfully.")
except Exception as e:
    print(f"Error initializing client: {e}")

In [ ]:
prompt = "Explain the concept of a Vector Database in two simple sentences."

try:
    response = client.predict(
        endpoint="databricks-dbrx-instruct",
        inputs={
            "messages": [
                {"role": "system", "content": "You are a helpful AI assistant."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.3,
            "max_tokens": 100
        }
    )
    
    print(f"\nPrompt: {prompt}")
    print(f"\nResponse: {response['choices'][0]['message']['content']}")
except Exception as e:
    print(f"Failed to query model: {e}")

## 3. AI Functions in Databricks SQL

Databricks allows you to call LLMs directly within SQL queries using the `ai_query` function. This is incredibly powerful for batch inference or ETL pipelines on Delta tables.

In [ ]:
# Example: Using ai_query to extract sentiment from customer reviews

sql_query = """
-- SELECT 
--   review_id,
--   review_text,
--   ai_query(
--     'databricks-dbrx-instruct',
--     CONCAT('Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL: ', review_text)
--   ) AS sentiment
-- FROM main.default.customer_reviews
-- LIMIT 5;
"""

print("Execute the following SQL in a Databricks SQL Warehouse or Notebook cell:")
print(sql_query)

## 4. Mosaic AI Vector Search

Vector Search automatically syncs Delta tables to a scalable vector index. This section demonstrates how to programmatically create an endpoint and sync an index.

In [ ]:
from databricks.vector_search.client import VectorSearchClient

# Initialize the Vector Search client
vsc = VectorSearchClient()

ENDPOINT_NAME = "my_rag_endpoint"
INDEX_NAME = "main.rag_catalog.documents_index"
SOURCE_TABLE = "main.rag_catalog.documents"

print(f"Creating/Checking endpoint: {ENDPOINT_NAME}")
try:
    # vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    # print(vsc.get_endpoint(ENDPOINT_NAME))
    print("Endpoint creation code is commented out to prevent accidental billing.")
except Exception as e:
    print(f"Endpoint error: {e}")
    
print(f"\nCreating/Syncing index: {INDEX_NAME} from {SOURCE_TABLE}")
try:
    # index = vsc.create_delta_sync_index(
    #     endpoint_name=ENDPOINT_NAME,
    #     index_name=INDEX_NAME,
    #     source_table_name=SOURCE_TABLE,
    #     pipeline_type="TRIGGERED",
    #     primary_key="doc_id",
    #     embedding_source_column="content",
    #     embedding_model_endpoint_name="databricks-bge-large-en"
    # )
    # print("Index created successfully.")
    print("Index creation code is commented out. Requires a valid Delta table in Unity Catalog.")
except Exception as e:
    print(f"Index error: {e}")

## 5. Building a RAG Chain with LangChain

Once your Vector Index is populated, you can use LangChain to build a Retrieval-Augmented Generation (RAG) pipeline.

In [ ]:
from langchain_community.vectorstores import DatabricksVectorSearch
from langchain_community.chat_models import ChatDatabricks
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

try:
    vector_store = DatabricksVectorSearch(
        endpoint=ENDPOINT_NAME,
        index_name=INDEX_NAME
    )
    
    llm = ChatDatabricks(endpoint="databricks-dbrx-instruct", max_tokens=200)
    
    prompt_template = """Use the following pieces of context to answer the question at the end.\n    If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n    Context: {context}\n\n    Question: {question}\n    Answer:"""
    PROMPT = PromptTemplate(
        template=prompt_template, input_variables=["context", "question"]
    )
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
        chain_type_kwargs={"prompt": PROMPT}
    )
    
    # question = "What is the company policy on remote work?"
    # answer = qa_chain.run(question)
    # print(f"Question: {question}\nAnswer: {answer}")
    print("RAG chain constructed successfully. Uncomment the run block to execute against a live index.")
    
except Exception as e:
    print(f"Error constructing RAG chain: {e}")

## 6. Fine-Tuning a Custom Model

Databricks provides a managed service to fine-tune Foundation Models on your private data using the `WorkspaceClient`.

In [ ]:
from databricks.sdk import WorkspaceClient

try:
    w = WorkspaceClient()
    
    print("Configuring a fine-tuning run for a custom Llama 3 model...")
    
    # run = w.model_training.create(
    #     name="custom-support-llama3",
    #     model="meta-llama/Meta-Llama-3-8B-Instruct",
    #     train_data_path="main.training_data.customer_support_qa",
    #     task_type="CHAT_COMPLETION",
    #     training_duration_epochs=3,
    #     learning_rate=5e-5
    # )
    # print(f"Fine-tuning started! Run ID: {run.id}")
    
    print("Fine-tuning code is commented out. Requires training data formatted in Unity Catalog.")
except Exception as e:
    print(f"Error initializing WorkspaceClient: {e}")